# 整合專題 (Capstone)：可控的多模型問答助理

## 模組脈絡：把控制權階梯整條走一遍

本專題整合 **01–07** 所有模組，打造一個**可控**的問答助理。它示範如何把「不可控的 LLM」透過一層層工程收斂成「可信賴、可驗收」的系統。

### 控制權階梯回顧（你已學會的收斂手段）
| 模組 | 收斂層 | 在本專題的角色 |
|------|--------|----------------|
| 01 不可控 | — | 理解 temperature/取樣為何不可控 |
| 02 意圖收斂 | prompt → spec | 用 spec 定義助理行為 |
| 03 結構收斂 | 結構化輸出 | Pydantic 驗證答案結構 |
| 04 知識收斂 | RAG | 把答案錨定在文件來源 |
| 05 行為收斂 | agent + guardrails | 工具呼叫 + prompt injection 防禦 |
| 06 協作收斂 | 多 agent | 跨模型互審把關 |
| 07 校準 | 評估 + 回饋 | 量測並持續改進 |

## 專題規格 (Spec)

**目標**：建一個對「指定文件集」回答問題的助理，需同時滿足：

1. **多模型並陳**：可切換 OpenAI / Claude / Gemini 作為生成模型（provider 抽象層）。
2. **RAG grounding**：答案必須基於檢索到的文件，無法回答時誠實說「不知道」。
3. **結構化輸出**：回傳 `{answer, citations, confidence}`，用 Pydantic 驗證。
4. **Guardrails**：輸入經 Moderation；防禦（間接）prompt injection。
5. **可評估**：附一組評估資料與指標（Factuality / Context Recall）。

**成功標準**：
- 在評估集上 Factuality ≥ 0.8；無檢索命中時正確回「不知道」率 ≥ 0.9。
- 切換三家 provider 皆能產生結構一致、可解析的輸出。
- 對注入攻擊測試（藏在文件中的惡意指令）不執行越權行為。

## 架構

```
使用者問題
  → [Moderation 輸入護欄]                (模組 05)
  → [RAG 檢索 top-k 文件]                 (模組 04)
  → [Provider 抽象層：OpenAI/Claude/Gemini] (模組 01,06)
  → [結構化輸出 answer/citations/confidence] (模組 03)
  → [低信心 → 觸發跨模型互審 / 人工]        (模組 06,07)
  → [記錄供回饋迴圈評估]                    (模組 07)
```

## 骨架程式（待學員補完 TODO）

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

from openai import OpenAI
from pydantic import BaseModel

openai_client = OpenAI()

class Answer(BaseModel):
    answer: str
    citations: list[str]
    confidence: float  # 0-1


In [ ]:
def moderate(text):
    """輸入護欄（模組 05）。"""
    r = openai_client.moderations.create(model="omni-moderation-latest", input=text)
    return r.results[0].flagged

def retrieve(question, k=5):
    """TODO：接上模組 04 的向量資料庫檢索，回傳文件片段列表。"""
    raise NotImplementedError("請接上你在模組 04 建立的 chromadb collection")

def generate(question, context, provider="openai"):
    """Provider 抽象層（模組 01/06）：依 provider 呼叫對應 SDK，回傳 Answer。
    TODO：openai 用 client.responses.parse(text_format=Answer)；
          claude 用 anthropic tool_use；gemini 用 response_schema。"""
    raise NotImplementedError


In [ ]:
def assistant(question, provider="openai"):
    # 1) 輸入護欄
    if moderate(question):
        return Answer(answer="很抱歉，無法處理此內容。", citations=[], confidence=1.0)
    # 2) RAG 檢索
    context = retrieve(question)
    # 3) 結構化生成（grounded）
    result = generate(question, context, provider=provider)
    # 4) 低信心 → TODO：觸發跨模型互審（模組 06）或人工
    if result.confidence < 0.5:
        pass  # TODO: cross_model_review(question, context)
    return result

# print(assistant("你的文件講了什麼？"))  # 接上 retrieve/generate 後即可執行

## 練習任務

1. 接上模組 04 的 chromadb 檢索，完成 `retrieve()`。
2. 完成 `generate()` 的三家 provider 分支（結構化輸出）。
3. 加入模組 06 的跨模型互審作為低信心 fallback。
4. 用模組 07 的評估流程量測 Factuality 與「不知道」率，達標準。
5. 設計 3 個間接 prompt injection 測試，確認助理不被攻破。

---

## 結語

這個專題把整條**控制權階梯**走了一遍：從理解不可控（01），到用 spec、結構、知識、行為、協作逐層收斂（02–06），最後用評估與回饋持續校準（07）。

> **核心信念**：你無法讓 LLM 變得確定，但你可以**測量它、約束它、驗證它**——這就是 AI 可控性工程。